# DhakaFlow P3 — Forecasting Gate (Colab)

**Needs: GPU runtime (T4 fine). No paid keys.** HF token only for the later
Chronos-2 arm (READY_FOR_KEY).

Runs the forecasting benchmark on the P2 `DHAKA_SIMULATED_SENSOR_GRAPH`
dataset (produced by `P2_sensor_graph_colab.ipynb`, on Drive).

Protocol (predeclared, enforced in `research/forecasting/baselines.py`):
chronological split; horizons 5/15/30 min (= 1/3/6 steps at 300 s);
MAE/RMSE always, MAPE only with its validity mask; every result labelled
SIMULATED. Floors (Persistence, Historical Average) come first; a learned
model that cannot beat them is REJECTED regardless of pedigree.

Harness decision (recorded in `upstream/UPSTREAM_PINS.yaml`): graph models
(DCRNN, GWNet, STID, STAEformer) run on the **legacy BasicTS benchmark
checkout** `4c3e2d3` — the published fair-comparison layout that contains
them all; the v1.0 checkout stays pinned for its API. Official code,
unmodified; we only add dataset files.

In [ ]:
# 1 · Repo + Drive
from google.colab import drive
drive.mount('/content/drive')
P2_DIR = '/content/drive/MyDrive/dhakaflow/artifacts/p2_sensor_graph'
RESULTS_DIR = '/content/drive/MyDrive/dhakaflow/artifacts/p3_forecasting'

REPO_DIR = '/content/thesis'
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/auth889-ai/thesis.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
import json, pathlib
m = json.loads(pathlib.Path(P2_DIR, 'MANIFEST.json').read_text())
assert m['evidence_domain'] == 'DHAKA_SIMULATED_SENSOR_GRAPH'
assert pathlib.Path(P2_DIR, 'P2_PASS_EVIDENCE.json').exists(), \
    'Run P2_sensor_graph_colab.ipynb to completion first (acceptance checks)'
print('P2 dataset OK:', m['shape'])

In [ ]:
# 2 · Floors FIRST (torch-free): Persistence + Historical Average.
!python scripts/run_forecast_baselines.py --data {P2_DIR} --out /content/baseline_results.json
import shutil, os
os.makedirs(RESULTS_DIR, exist_ok=True)
shutil.copy('/content/baseline_results.json', RESULTS_DIR + '/baseline_results.json')

In [ ]:
# 3 · Legacy BasicTS harness at the pinned SHA (official, unmodified).
BTS = '/content/BasicTS_legacy'
LEGACY_SHA = '4c3e2d373c647667c2123e1591f2d79c229dfcc6'
if not os.path.exists(BTS):
    subprocess.run(['git','clone','https://github.com/GestaltCogTeam/BasicTS.git', BTS], check=True)
subprocess.run(['git','-C',BTS,'checkout',LEGACY_SHA], check=True)
%pip -q install easytorch easydict
print('legacy harness at', LEGACY_SHA)

In [ ]:
# 4 · Export the P2 dataset into BOTH harness formats.
from research.forecasting.basicts_adapter import (
    export_basicts_dataset, export_basicts_legacy_dataset)
import pathlib
legacy_dir = pathlib.Path(BTS, 'datasets', 'DhakaR1SensorGraph_legacy')
print(export_basicts_legacy_dataset(pathlib.Path(P2_DIR), legacy_dir))
print(export_basicts_dataset(pathlib.Path(P2_DIR), pathlib.Path('/content/dhaka_v1_dataset')))

## 5 · Graph-model runs (template)

For each candidate, copy its METR-LA config from `baselines/<MODEL>/` to a
`DhakaR1SensorGraph_legacy.py` variant changing ONLY (predeclared):
`DATA_NAME`, `INPUT_LEN = 12` (1 h history), `OUTPUT_LEN = 6` (30 min —
horizons 1/3/6 reported from the same run), ratios `[0.7, 0.1, 0.2]`,
`NULL_VAL = 0.0`, num_nodes/adjacency path. Model hyperparameters stay the
authors' METR-LA values (recorded as OFFICIAL; any change must be labelled
ASSUMED with a reason). One candidate per cell; capture wall-clock + VRAM.
Order: STID (cheapest) → GWNet → DCRNN → STAEformer.

In [ ]:
# 5a · STID example (first candidate).
# 1) create the config variant as described above, then:
#    %cd {BTS}
#    !python experiments/train.py -c baselines/STID/DhakaR1SensorGraph_legacy.py -g 0
# 2) copy the result json/ckpt summary to RESULTS_DIR with the split hash.
print('Template — fill per candidate at gate time; no winner before all runs.')